# Etapa 1 — Smoke Test Conjunto de Arquitecturas Candidatas (`all_models_smoke`)

Replica `tools/smoke_all_models.py` en `PINN_V4` evaluando todo el abanico candidato antes de invertir tiempo de cómputo largo:
- **Modelos evaluados**: `M0`, `M1`, `M2`, `DCSNN`, `M0R`, `CUSP`, `CUSPR`, `FBPINN`, `M1_EXACT`, `CUSP_EXACT`.
- **Casos pivote**: `mms_sine` (analítico suave) y `cable_disk` (concentración energética extrema en conductor + contraste $k$).
- **Objetivo**: verificar estabilidad numérica, ausencia de NaNs y obtener señales iniciales de gradientes/escala.

In [1]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "full_campaign"    #"demo_fast"

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(12, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='full_campaign' | USE_PARALLEL=True (workers=12, device='cpu') | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
CASES_STAGE1 = ("mms_sine", "cable_disk")
CANDIDATE_MODELS = (
    ("M0", "M1", "M2", "CUSPR", "M1_EXACT")
    if RUN_MODE == "demo_fast"
    else ("M0", "M1", "M2", "DCSNN", "M0R", "CUSP", "CUSPR", "FBPINN", "M1_EXACT", "CUSP_EXACT")
)
STEPS_STAGE1 = 250 if RUN_MODE == "demo_fast" else 1200
WIDTH_STAGE1 = 24
DEPTH_STAGE1 = 2

In [3]:
stage1_dir = CAMP_WS / "stage1_smoke_models"
stage1_dir.mkdir(parents=True, exist_ok=True)
stage1_summary = {}
tasks_s1 = []

for case_id in CASES_STAGE1:
    prob = catalog.load(case_id)
    ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=False)
    for arch in CANDIDATE_MODELS:
        key = f"{case_id}:{arch}"
        run_dir = stage1_dir / case_id / arch
        tasks_s1.append(
            RunTask(
                name=key,
                problem_source=prob.doc,
                architecture=arch,
                output_dir=run_dir,
                config_kwargs={
                    "adam_steps": STEPS_STAGE1,
                    "lbfgs_steps": 10,
                    "lr": 2e-3,
                    "width": WIDTH_STAGE1,
                    "depth": DEPTH_STAGE1,
                    "weights": {"pde": 1.0, "energy": 100.0},
                    "sampling": "mixed",
                    "focused": True,
                    "focus_criterion": "residual",
                    "focus_surfaces": False,
                    "resample_every": 100,
                    "full_audit": True,
                    "dtype": "float32",
                    "device": DEVICE,
                },
                reference=ref_vals,
                reference_label=ref_lbl,
                overwrite=True,
            )
        )

def _on_stage1_done(res: dict) -> None:
    key = res["name"]
    case_id, arch = key.split(":", 1)
    if res.get("status") == "completed":
        m = extract_metrics(res)
        stage1_summary[key] = {
            "case": case_id,
            "architecture": arch,
            "status": res.get("status"),
            "best_loss": res.get("best_loss"),
            **m,
        }
        loss_str = f"{res['best_loss']:.3e}" if res.get("best_loss") is not None else "nan"
        tmax_str = f"{m['Tmax_pred_C']:.2f} C" if m.get("Tmax_pred_C") is not None else "n/a"
        print(f"[Etapa 1] {key:24s} | dev={res.get('device')} | status={res.get('status')} | loss={loss_str} | Tmax={tmax_str}")
    else:
        stage1_summary[key] = {"case": case_id, "architecture": arch, "status": "failed", "error": str(res.get("error"))}
        print(f"[Etapa 1] {key:24s} | FAILED: {res.get('error')}")
    write_json_atomic(CAMP_WS / "stage1_smoke_summary.json", stage1_summary)

runner_s1 = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
runner_s1.run(tasks_s1, on_complete=_on_stage1_done, show_progress=True, progress_title="Etapa 1 — Smoke Arquitecturas")
write_json_atomic(CAMP_WS / "stage1_smoke_summary.json", stage1_summary)

C:\Users\QU1267\AppData\Local\Temp\ipykernel_42404\38793172.py:61: RuntimeWarning: El pool de 12 procesos se interrumpió (BrokenProcessPool); reintentando 20 tarea(s) pendientes con 6 trabajador(es).
  runner_s1.run(tasks_s1, on_complete=_on_stage1_done)


[Etapa 1] mms_sine:M2              | dev=cpu | status=completed | loss=1.646e-01 | Tmax=24.54 C
[Etapa 1] mms_sine:M1              | dev=cpu | status=completed | loss=5.653e-02 | Tmax=51.54 C
[Etapa 1] mms_sine:CUSP            | dev=cpu | status=completed | loss=5.653e-02 | Tmax=51.54 C
[Etapa 1] mms_sine:DCSNN           | dev=cpu | status=completed | loss=3.194e-02 | Tmax=49.83 C
[Etapa 1] mms_sine:M0              | dev=cpu | status=completed | loss=6.126e-03 | Tmax=49.44 C
[Etapa 1] mms_sine:M0R             | dev=cpu | status=completed | loss=3.107e-02 | Tmax=49.08 C
[Etapa 1] mms_sine:CUSPR           | dev=cpu | status=completed | loss=3.107e-02 | Tmax=49.08 C
[Etapa 1] mms_sine:M1_EXACT        | dev=cpu | status=completed | loss=7.228e-07 | Tmax=50.00 C
[Etapa 1] cable_disk:M1            | dev=cpu | status=completed | loss=5.865e-03 | Tmax=21.18 C
[Etapa 1] mms_sine:CUSP_EXACT      | dev=cpu | status=completed | loss=7.228e-07 | Tmax=50.00 C
[Etapa 1] cable_disk:M0            | dev

WindowsPath('C:/usr/ths_mia_fiis/tesis_MIA_2026-2_pinn/notebooks_campaign/campaign_workspace/stage1_smoke_summary.json')